# Entrenamiento independiente de B4

Este cuaderno ejecuta solo B4 para los escenarios A, B y C con todas las partidas de `train`. Guarda estudios, métricas y modelos en `MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente/`. Sus checkpoints son independientes de B3 y del pipeline secuencial. Si Colab se interrumpe, vuelve a ejecutar setup y entrenamiento para reanudar. `val` compara finalistas; `test` no se toca.

Para empezar, la persona que lo ejecute debe iniciar sesión en Google, permitir el montaje de su propio Drive y, si el dataset no existe en `/content/datos`, configurar sus credenciales/permisos de Kaggle para descargarlo. Abrir el enlace por sí solo no inicia el entrenamiento.

In [ ]:
from pathlib import Path
import sys
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
from sklearn import __version__ as sklearn_version
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'B4 necesita scikit-learn >=1.7; reinicia el runtime.'
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa Kaggle y vuelve a ejecutar setup.'
print('Entorno B4 listo | sklearn:', sklearn_version, '| checkpoints:', CHECKPOINT_ROOT)

## Presupuesto de búsqueda

- **Una sola búsqueda** de hiperparámetros (Optuna, TPE) para los tres escenarios, con validación cruzada de 3 pliegues agrupada por partida.
- Solo usa partidas que son `train` en A, B y C a la vez: los hiperparámetros nunca ven partidas de `val`, `calib` o `test` de ningún escenario.
- Un solo estudio optimiza log loss (regla de puntuación propia). AUC y Brier se registran en cada trial, y las finalistas por cada métrica salen del mismo estudio.
- 20 trials, con `max_iter = 2000` y parada temprana (paciencia 30) sobre un grupo de vigilancia separado.
- Sin límite de tiempo: si Colab se corta, al volver a ejecutar se reanuda desde el último pliegue guardado en Drive.
- Después, cada escenario reentrena las finalistas con **todo** su `train` y las compara en su `val`. `test` no se toca.
- El materializado de datos B4 se reconstruye en el disco local de cada runtime.

In [ ]:
# B4 - BUSQUEDA UNICA DE HIPERPARAMETROS Y FINALISTAS POR ESCENARIO
# PROPOSITO:
# 1) Buscar hiperparametros UNA sola vez (Optuna TPE, validacion cruzada de 3 pliegues agrupada
#    por partida) usando SOLO partidas que son train en A, B y C a la vez. Asi ningun
#    hiperparametro vio partidas de val, calib o test de ningun escenario.
# 2) Cada escenario reentrena las finalistas con TODO su train y las compara en su val.
# - Un solo estudio optimiza log loss; AUC y Brier de cada trial quedan registrados y definen
#   las finalistas sin ajustes extra.
# - max_iter alto + parada temprana: el numero de arboles lo decide la vigilancia agrupada.
# - Todo se guarda en Drive por pliegue: si Colab se corta, volver a ejecutar reanuda.
# ESPERADO: "nucleos de CPU" y unas 11 000 partidas de busqueda; lineas
# "pliegue k/3: checkpoint guardado | arboles=N" con N menor que 2000 (si siempre es 2000,
# subir B4_MAX_ITER); al final b4_candidatas.csv y un b4_val_finalistas.csv por escenario.
# test NO se toca.
import os
from tempfile import TemporaryDirectory
import gc
import faulthandler
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b4 import materialize_split, b4_optuna_cv, fit_b4_candidates_val
from ptcg_wp.checkpoints import atomic_csv
from ptcg_wp.features import feature_columns

features = feature_columns(pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names)
B4_TRIALS = 20
B4_SPLITS = 3
B4_SEARCH_FRACTION = 1.0   # bajar a 0.5 si un trial tarda mas de ~40 min
B4_MAX_ITER = 2000
B4_PATIENCE = 30
B4_OBJECTIVE = 'log_loss'
MAX_RAM_GIB = 10.0
SEED = 2026
RUN_ROOT = CHECKPOINT_ROOT / 'busqueda_v3'
CANDIDATES_PATH = RUN_ROOT / 'busqueda' / 'b4_candidatas.csv'
print('nucleos de CPU:', os.cpu_count())

# Partidas que son train en los tres escenarios: las unicas que ve la busqueda
part = pd.read_parquet(DATA_DIR / 'particiones.parquet')
comunes = part.loc[(part['split_A'] == 'train') & (part['split_B'] == 'train')
                   & (part['split_C'] == 'train'), 'game_id']
print('partidas de busqueda (train en A, B y C):', len(comunes))

trace_file = open('/content/b4_pipeline_tracebacks.log', 'a', buffering=1)
faulthandler.enable(file=trace_file)
faulthandler.dump_traceback_later(600, repeat=True, file=trace_file)
b4_outputs = {}
try:
    # 1) Busqueda unica (se omite si ya termino en una sesion anterior)
    if not CANDIDATES_PATH.exists():
        print('B4: busqueda de hiperparametros', flush=True)
        with TemporaryDirectory(prefix='b4-busqueda-', dir='/content') as cache:
            stream = ScenarioStream(DATA_DIR, 'A', tuple(features), batch_size=32768)
            train = materialize_split(stream, features, 'train', cache)
            try:
                codigos = np.flatnonzero(train.games.isin(comunes))
                _, candidates = b4_optuna_cv(
                    train, n_splits=B4_SPLITS, n_trials=B4_TRIALS, timeout=None, seed=SEED,
                    max_iter=B4_MAX_ITER, patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=RUN_ROOT / 'busqueda',
                    study_cache_dir='/content/ptcg_optuna/b4_busqueda_v3',
                    verbose=True, objective_metric=B4_OBJECTIVE,
                    search_fraction=B4_SEARCH_FRACTION, search_groups=codigos,
                )
                atomic_csv(candidates.to_dict('records'), CANDIDATES_PATH)
            finally:
                train.close()
            del stream
        gc.collect()

    # Se leen siempre del CSV para que la reanudacion use exactamente las mismas finalistas
    candidates = pd.read_csv(CANDIDATES_PATH)
    print(candidates.to_string(index=False), flush=True)

    # 2) Finalistas reentrenadas con todo el train de cada escenario y comparadas en su val
    for scenario in ('A', 'B', 'C'):
        output_dir = RUN_ROOT / f'escenario_{scenario}'
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f'B4 {scenario}: preparando train y val', flush=True)
        with TemporaryDirectory(prefix=f'b4-{scenario}-', dir='/content') as cache:
            stream = ScenarioStream(DATA_DIR, scenario, tuple(features), batch_size=32768)
            train = materialize_split(stream, features, 'train', cache)
            val = materialize_split(stream, features, 'val', cache)
            try:
                val_metrics = fit_b4_candidates_val(
                    train, val, candidates, seed=SEED, max_iter=B4_MAX_ITER,
                    patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=output_dir, scenario=scenario, verbose=True,
                )
                atomic_csv(val_metrics.to_dict('records'), output_dir / 'b4_val_finalistas.csv')
                b4_outputs[scenario] = str(output_dir)
                print(f'B4 {scenario}: terminado | finalistas={len(candidates)} | {output_dir}', flush=True)
                del val_metrics, stream
            finally:
                train.close()
                val.close()
        gc.collect()
finally:
    faulthandler.cancel_dump_traceback_later()
    faulthandler.disable()
    trace_file.close()
print('B4 listo. Checkpoints:', b4_outputs)